# 02 · Subset flux columns

Reads the per-version Parquet files produced in
[notebook 01](01_read_fluxes_to_parquet.ipynb), keeps only a defined list of
columns, restricts the rows to the analysis year (`YEAR`), and writes the reduced
tables back out as Parquet. This keeps the downstream comparison focused on the
variables of interest (fluxes, time-lag diagnostics, and potential shortwave
radiation for a later daytime / nighttime split) instead of the full
~483-column EddyPro output.

All five scenarios (`*-1` through `*-5`) are processed here, ten files across
both analyzers. Files are discovered by glob, so new versions need no code change.

- Input: `data/01-eddypro_fluxes_level-1_parquet/`, the full per-version tables.
- Output: `data/02-eddypro_fluxes_level-1_parquet_subsets/`, the column subsets
  for the analysis year.

The set of columns kept is defined explicitly in `KEEP_COLS` below so it stays
auditable alongside the [processing versions](../docs/processing-versions.md).
For the `*-5` (PWB) scenario the lag was detected and removed from the raw data
before flux processing, so its `*_TLAG_USED` columns carry no compensation;
the actual per-chunk PWB time lags come from the separate `*_pwb_tlag.parquet`.

## Imports

In [1]:
from datetime import datetime
from pathlib import Path

from diive.core.io.files import load_parquet, save_parquet

NB_START = datetime.now()  # notebook start time (reported in the last cell)

## Select columns

In [2]:
# Columns to keep in each subset. Edit this list as the analysis requires.
KEEP_COLS = [
    # Fluxes
    "FN2O",
    "FCH4",
    # N2O time-lag diagnostics
    "N2O_TLAG_USED",
    # CH4 time-lag diagnostics
    "CH4_TLAG_USED",
    # Potential shortwave radiation, for splitting fluxes into daytime
    # (SW_IN_POT > 0) and nighttime (SW_IN_POT == 0) later on.
    "SW_IN_POT",
]
print(f"{len(KEEP_COLS)} columns requested.")

5 columns requested.


## Configuration

In [3]:
# Input / output folders (relative to the notebooks/ directory).
INDIR = Path("../data/01-eddypro_fluxes_level-1_parquet")
OUTDIR = Path("../data/02-eddypro_fluxes_level-1_parquet_subsets")
OUTDIR.mkdir(parents=True, exist_ok=True)

# This analysis uses only one year; rows outside it are dropped from the subsets.
YEAR = 2021

# Discover the input Parquet files.
parquet_files = sorted(INDIR.glob("*.parquet"))
print(f"Found {len(parquet_files)} Parquet file(s):")
for f in parquet_files:
    print(f"  {f.name}")

Found 10 Parquet file(s):
  LGR-1.parquet
  LGR-2.parquet
  LGR-3.parquet
  LGR-4.parquet
  LGR-5.parquet
  QCL-1.parquet
  QCL-2.parquet
  QCL-3.parquet
  QCL-4.parquet
  QCL-5.parquet


## Subset each file and save

For each version: load the full table, keep `KEEP_COLS` (preserving the
timestamp index), warn about any requested columns that are absent, and save the
subset under the same version-code filename.

In [4]:
saved = {}
for f in parquet_files:
    code = f.stem  # version code, e.g. 'LGR-1'
    print(f"\n=== {code} ===")

    df = load_parquet(filepath=str(f))

    # Keep only the analysis year.
    df = df.loc[df.index.year == YEAR]

    present = [c for c in KEEP_COLS if c in df.columns]
    missing = [c for c in KEEP_COLS if c not in df.columns]
    if missing:
        print(f"  WARNING: {len(missing)} requested column(s) not found: {missing}")

    subset = df[present]
    print(f"  kept {subset.shape[1]} / {len(KEEP_COLS)} columns, "
          f"{subset.shape[0]} rows in {YEAR}")

    filepath = save_parquet(filename=code, data=subset, outpath=str(OUTDIR))
    saved[code] = filepath


=== LGR-1 ===


> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\LGR-1.parquet (0.086 seconds).

  kept 5 / 5 columns, 7803 rows in 2021


> Saved file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-1.parquet (0.008 seconds).


=== LGR-2 ===


...

> Saved file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-4.parquet (0.014 seconds).


=== QCL-5 ===


> Loaded .parquet file ..\data\01-eddypro_fluxes_level-1_parquet\QCL-5.parquet (0.066 seconds).

  kept 5 / 5 columns, 9632 rows in 2021


> Saved file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-5.parquet (0.010 seconds).

## Verify

Reload one subset to confirm the round-trip and the retained columns.

In [5]:
print("Saved subset files:")
for code, path in saved.items():
    print(f"  {code}: {path}")

check = load_parquet(filepath=next(iter(saved.values())))
print(f"\nColumns ({check.shape[1]}): {list(check.columns)}")
check.describe()

Saved subset files:
  LGR-1: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-1.parquet
  LGR-2: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-2.parquet
  LGR-3: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-3.parquet
  LGR-4: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-4.parquet
  LGR-5: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-5.parquet
  QCL-1: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-1.parquet
  QCL-2: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-2.parquet
  QCL-3: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-3.parquet
  QCL-4: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-4.parquet
  QCL-5: ..\data\02-eddypro_fluxes_level-1_parquet_subsets\QCL-5.parquet


> Loaded .parquet file ..\data\02-eddypro_fluxes_level-1_parquet_subsets\LGR-1.parquet (0.003 seconds).


Columns (5): ['FN2O', 'FCH4', 'N2O_TLAG_USED', 'CH4_TLAG_USED', 'SW_IN_POT']


,FN2O,FCH4,N2O_TLAG_USED,CH4_TLAG_USED,SW_IN_POT
count,7729.000000,7729.000000,7729.000000,7729.000000,7755.000000
mean,1.836282,15.381377,3.644935,4.452788,256.190261
std,9.188328,221.569709,3.104414,3.659447,346.888345
min,-374.774000,-7207.160000,-0.050000,-0.050000,0.000000
25%,0.116048,-11.453500,1.650000,1.650000,0.000000
50%,0.725968,4.458190,1.950000,2.750000,0.000000
75%,2.552340,26.863700,5.950000,8.400000,480.500500
max,205.036000,5272.880000,10.000000,10.000000,1187.430000


## Runtime

In [6]:
NB_END = datetime.now()
print(f"Start:    {NB_START:%Y-%m-%d %H:%M:%S}")
print(f"End:      {NB_END:%Y-%m-%d %H:%M:%S}")
print(f"Runtime:  {NB_END - NB_START}")

Start:    2026-06-24 15:17:07
End:      2026-06-24 15:17:09
Runtime:  0:00:02.035606
